In [1]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# requires `regex` package
import regex as re
re.findall(PAT, "some text that i'll pre-tokenize")


['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [2]:
for word in re.finditer(PAT, "some text that i'll pre-tokenize"):
    print(word)

<regex.Match object; span=(0, 4), match='some'>
<regex.Match object; span=(4, 9), match=' text'>
<regex.Match object; span=(9, 14), match=' that'>
<regex.Match object; span=(14, 16), match=' i'>
<regex.Match object; span=(16, 19), match="'ll">
<regex.Match object; span=(19, 23), match=' pre'>
<regex.Match object; span=(23, 24), match='-'>
<regex.Match object; span=(24, 32), match='tokenize'>


In [3]:
doc = '''low low low low low
lower lower widest widest widest
newest newest newest newest newest newest'''


In [4]:
for word in re.finditer(PAT, "some text that i'll pre-tokenize"):
  break
word

<regex.Match object; span=(0, 4), match='some'>

In [5]:
from collections import Counter

pre_tokens = (tuple(item.group()) for item in re.finditer(r'\S+', doc))
ctr = Counter(pre_tokens)
ctr

Counter({('n', 'e', 'w', 'e', 's', 't'): 6,
         ('l', 'o', 'w'): 5,
         ('w', 'i', 'd', 'e', 's', 't'): 3,
         ('l', 'o', 'w', 'e', 'r'): 2})

In [11]:
for key in ctr.keys():
  break
type(key[0])

str

In [17]:
chr(0), print(chr(0), end=' '), ord('א')

  

('\x00', None, 1488)

In [29]:
str = 'abcd אבגד'
encoded = str.encode('utf-8')
type(encoded), type(encoded[0])
byte_encoded = bytes(encoded)
byte_encoded

b'abcd \xd7\x90\xd7\x91\xd7\x92\xd7\x93'

In [34]:
'BA' > 'AB', 'BA'.encode('utf-8') > 'AB'.encode('utf-8'), 'BA'.encode('utf-8'), 'AB'.encode('utf-8')
'BC'.encode('utf-8') > 'BAA'.encode('utf-8')

True

In [54]:
from tokenizer import get_corpus_it

%time corpus = Counter(get_corpus_it('tests/fixtures/corpus.en', ['<|endoftext|>']))

CPU times: user 156 ms, sys: 1.5 ms, total: 157 ms
Wall time: 156 ms


In [58]:
len(corpus.keys()) #, 
print(corpus, end=' ')

Counter({(b' ', b','): 1306, (b' ', b't', b'h', b'e'): 1279, (b'\n',): 1015, (b' ', b'.'): 962, (b' ', b'o', b'f'): 646, (b' ', b'a', b'n', b'd'): 609, (b' ', b'a'): 480, (b' ', b't', b'o'): 474, (b' ', b'i', b'n'): 436, (b';',): 357, (b' ', b'i', b's'): 338, (b' ', b'&'): 270, (b' ', b'f', b'o', b'r'): 237, (b' ', b'y', b'o', b'u'): 232, (b' ', b't', b'h', b'a', b't'): 207, (b' ', b'b', b'e'): 173, (b' ', b'@', b'-', b'@'): 169, (b' ', b'o', b'n'): 156, (b' ', b'i', b't'): 151, (b' ', b'a', b'r', b'e'): 151, (b' ', b'w', b'i', b't', b'h'): 148, (b'q', b'u', b'o', b't'): 146, (b' ', b'a', b's'): 140, (b' ', b't', b'h', b'i', b's'): 127, (b' ', b';'): 123, (b' ', b'f', b'r', b'o', b'm'): 116, (b'a', b'p', b'o', b's'): 116, (b' ', b'I'): 114, (b' ', b'('): 113, (b' ', b')'): 108, (b' ', b'b', b'y'): 105, (b' ', b'y', b'o', b'u', b'r'): 101, (b' ', b'b'): 100, (b' ', b'o', b'r'): 98, (b' ', b'n', b'o', b't'): 98, (b' ', b't', b'h', b'e', b'y'): 96, (b' ', b'c', b'a', b'n'): 95, (b' ', b'w

In [41]:
%%time
from collections import defaultdict
vocab, freqs = {}, {}
pair_counts = Counter()
pair_words = defaultdict(set)

for word_id, (tokens, freq) in enumerate(corpus.items()):
  vocab[word_id] = list(tokens)
  freqs[word_id] = freq

CPU times: user 4.3 ms, sys: 0 ns, total: 4.3 ms
Wall time: 4.07 ms


In [42]:
%%time
for a, b in zip(tokens, tokens[1:]):
  pair_counts[(a, b)] += freq
  pair_words[(a, b)].add(word_id)


CPU times: user 70 μs, sys: 0 ns, total: 70 μs
Wall time: 72.7 μs


In [43]:
%%time
from pair_heap import PairHeap
pair_heap = PairHeap(pair_counts)

CPU times: user 41 μs, sys: 0 ns, total: 41 μs
Wall time: 44.3 μs


In [47]:
%%time
from tokenizer import apply_merge
merges = []
num_merges = 5000 - 256

for _ in range(num_merges):
  best_pair, count = pair_heap.pop_best()
  if best_pair is None or count == 0:
    break
  apply_merge(best_pair, vocab, freqs, pair_words, pair_heap)
  merges.append(best_pair)


CPU times: user 1.47 ms, sys: 0 ns, total: 1.47 ms
Wall time: 1.37 ms


In [53]:
len(pair_counts), print(pair_counts, end=' ')

Counter({(b' ', b'G'): 15, (b' G', b'e'): 15, (b' Ge', b's'): 15, (b' Ges', b'c'): 15, (b' Gesc', b'h'): 15, (b' Gesch', b'\xc3'): 15, (b' Gesch\xc3', b'\xa4'): 15, (b' Gesch\xc3\xa4', b'f'): 15, (b' Gesch\xc3\xa4f', b't'): 15, (b' Gesch\xc3\xa4ft', b's'): 15, (b' Gesch\xc3\xa4fts', b'o'): 15, (b' Gesch\xc3\xa4ftso', b'r'): 15, (b' Gesch\xc3\xa4ftsor', b'd'): 15, (b' Gesch\xc3\xa4ftsord', b'n'): 15, (b' Gesch\xc3\xa4ftsordn', b'u'): 15, (b' Gesch\xc3\xa4ftsordnu', b'n'): 15, (b' Gesch\xc3\xa4ftsordnun', b'g'): 15}) 

(17, None)

In [159]:
import regex as re
from typing import Iterable, Iterator, Any

def build_pattern(special_tokens: list[str]) -> re.Pattern:
    base = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    if not special_tokens:
        return re.compile(f"($^)|" + base)  # ($^) never matches anything
    sorted_specials = sorted(special_tokens, key=len, reverse=True)
    special_part = "|".join(re.escape(tok) for tok in sorted_specials)
    return re.compile(f"({special_part})|" + base)

def to_bytes(string: str) -> tuple[bytes, ...]:
  return tuple([bytes([b]) for b in bytes(string, encoding='utf-8')])

def read_corpus(text: Iterable, pat: re.Pattern) -> Iterator[str]:
   def _read_corpus(line, pat):
      for match in pat.finditer(line):
         if match.group(1):
            continue
         yield to_bytes(match.group())
   if isinstance(text, type(str)):
      yield from _read_corpus(text, pat)
   for line in text:
      yield from _read_corpus(line, pat)

def get_corpus(corpus_it: Iterator[tuple[bytes]]) -> Any:
   freq_table = Counter(corpus_it)
   # pair_words = defaultdict
   return freq_table

In [ ]:
import time

def test_to_bytes():
  assert to_bytes('abcd אבגד') == (
    b'a', b'b', b'c', b'd', b' ', b'\xd7', b'\x90', b'\xd7', b'\x91', b'\xd7', b'\x92', b'\xd7', b'\x93') 

def test_get_corpus():
  freq_table = get_corpus(read_corpus(
    '''low low low low low
    lower lower widest widest widest
    newest newest newest newest newest newest''',
    build_pattern(['<|endoftext|>'])
  ))
  assert freq_table[to_bytes(' newest')] == 6
  assert freq_table[to_bytes(' low')] == 4
  assert freq_table[to_bytes(' lower')] == 2
  assert freq_table[to_bytes('low')] == 1

def test_reading_time():
  now = time.time()
  with open('tests/fixtures/corpus.en', encoding='utf-8') as file:
    freq_table = get_corpus(read_corpus(file, build_pattern(['<|endoftext|>'])))
  assert len(freq_table) > 4000
  assert time.time() - now < 0.5

test_to_bytes()
test_get_corpus()
test_reading_time()

In [158]:
isinstance('''hello''', type(str))

True